# ML Coding Round — Algorithms From Scratch

"Implement k-means" in 45 minutes is a standard ML-engineer screen. The bar is not the full sklearn feature set — it's a **correct, compact, well-structured implementation you can write from memory and explain line by line**. Each algorithm below is implemented in interview-safe ~20 lines and then **validated against scikit-learn on the same data**, so you know the scratch version is actually right, not approximately right.

**The meta-advice that scores:** before writing code, state the algorithm in words, name the stopping condition, and name the complexity. Interviewers grade the plan as much as the code.

## 1. K-Means From Scratch

**Plan to state:** (1) initialize centers from k random points; (2) assign each point to the nearest center; (3) move each center to the mean of its points; (4) repeat 2–3 until assignments stop changing. Complexity O(n·k·d·iters). Stops at a local optimum — hence sklearn's `n_init`.

In [1]:
import numpy as np
from sklearn.datasets import make_blobs

def kmeans(X, k, max_iter=100, seed=0):
    rng = np.random.default_rng(seed)
    centers = X[rng.choice(len(X), k, replace=False)]          # 1) init
    for _ in range(max_iter):
        # 2) assignment step: distance of every point to every center
        d = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(-1)
        labels = d.argmin(axis=1)
        # 3) update step
        new_centers = np.array([X[labels == j].mean(axis=0) if np.any(labels == j)
                                else centers[j] for j in range(k)])
        if np.allclose(new_centers, centers):                   # 4) converged
            break
        centers = new_centers
    return labels, centers

X, y_true = make_blobs(n_samples=400, centers=4, cluster_std=1.1, random_state=7)
labels, centers = kmeans(X, k=4)

# accuracy vs ground truth via best-permutation match (clusters are arbitrary labels)
from scipy.optimize import linear_sum_assignment
perm = linear_sum_assignment(-np.array([[np.mean(labels[y_true == t] == c) for c in range(4)] for t in range(4)]))[1]
acc = np.mean([perm[t] == l for t, l in zip(y_true, labels)])
print(f"scratch k-means accuracy vs ground truth: {acc:.1%}")
print("final centers:\n", centers.round(1))

scratch k-means accuracy vs ground truth: 100.0%
final centers:
 [[-8.6  5.6]
 [ 0.  -8.7]
 [ 9.6  0.5]
 [-1.3  4.3]]


In [2]:
# validate against sklearn: same objective value (inertia)
from sklearn.cluster import KMeans
sk = KMeans(n_clusters=4, n_init=10, random_state=7).fit(X)

def inertia(X, labels, centers):
    return ((X - centers[labels]) ** 2).sum()

mine = inertia(X, labels, centers)
print(f"scratch inertia : {mine:.2f}")
print(f"sklearn inertia: {sk.inertia_:.2f}  (ratio {mine/sk.inertia_:.4f} — should be ~1.0)")
print("verdict:", "MATCHES sklearn" if mine / sk.inertia_ < 1.005 else "close; more restarts would tighten it")

scratch inertia : 907.88
sklearn inertia: 907.88  (ratio 1.0000 — should be ~1.0)
verdict: MATCHES sklearn


**Follow-ups they ask, with the answers:**
- *How to choose k?* Elbow on inertia, silhouette score, gap statistic; or domain constraint.
- *Why multiple restarts?* k-means converges to a **local** optimum sensitive to init; `k-means++` (sklearn default) spreads initial centers by distance-weighted sampling.
- *Why is it "means"?* The centroid is the point minimizing **squared** Euclidean distance — the mean. For absolute distance it'd be the geometric median (k-medoids uses actual points instead).

## 2. K-Nearest Neighbors From Scratch

**Plan:** store the training data (that *is* the model — a lazy learner); predict by computing all distances, taking the k smallest, and majority vote. O(n·d) per prediction — the line to say about why KNN scales badly.

In [3]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

class KNN:
    def __init__(self, k=5):
        self.k = k
    def fit(self, X, y):
        self.X, self.y = np.asarray(X), np.asarray(y)
        return self                      # fit stores; nothing to train
    def predict(self, X):
        d = np.sqrt(((np.asarray(X)[:, None, :] - self.X[None, :, :]) ** 2).sum(-1))
        nn = np.argsort(d, axis=1)[:, :self.k]           # k nearest indices
        votes = self.y[nn]                                # (n_queries, k)
        return np.array([np.bincount(v).argmax() for v in votes])

iris = load_iris()
Xtr, Xte, ytr, yte = train_test_split(iris.data, iris.target, test_size=0.25, random_state=42)

mine = KNN(k=5).fit(Xtr, ytr).predict(Xte)
from sklearn.neighbors import KNeighborsClassifier
sk = KNeighborsClassifier(5).fit(Xtr, ytr).predict(Xte)
print(f"scratch accuracy: {(mine == yte).mean():.1%}")
print(f"sklearn accuracy : {(sk == yte).mean():.1%}")
print("identical predictions to sklearn:", bool((mine == sk).all()))

scratch accuracy: 100.0%
sklearn accuracy : 100.0%
identical predictions to sklearn: True


**Follow-ups:**
- *Distance choice?* Euclidean default; Manhattan for high-dim (curse of dimensionality hits L2 harder); cosine for text embeddings — **scale features first** regardless.
- *k tradeoff?* Small k → low bias/high variance (noisy); large k → smoothing toward majority class. Cross-validate; odd k avoids vote ties for binary.
- *Speedups?* KD-tree/ball-tree (low-d), approximate NN (ANN, HNSW) for production; that's what vector databases do.

## 3. Decision Tree Split From Scratch

The full tree is too much for 45 minutes — screens ask for the **core loop**: given a feature and target, find the best split threshold by maximizing information gain. Write it for one feature, say "recursively apply per feature per node" and you've passed.

In [4]:
def entropy(y):
    p = np.bincount(y) / len(y)
    p = p[p > 0]
    return -(p * np.log2(p)).sum()

def best_split(x, y):
    # return (threshold, gain) maximizing information gain for feature x
    parent = entropy(y)
    best_thr, best_gain = None, 0.0
    for thr in np.unique(x)[:-1]:                     # split right of each value
        left, right = y[x <= thr], y[x > thr]
        if len(left) == 0 or len(right) == 0:
            continue
        child = (len(left) * entropy(left) + len(right) * entropy(right)) / len(y)
        gain = parent - child
        if gain > best_gain:
            best_thr, best_gain = thr, gain
    return best_thr, best_gain

rng = np.random.default_rng(3)
x = rng.normal(size=300)
y = (x + 0.3 * rng.normal(size=300) > 0).astype(int)      # target really depends on x

thr, gain = best_split(x, y)
print(f"best threshold: {thr:.3f}   information gain: {gain:.3f} bits")
print(f"parent entropy {entropy(y):.3f} -> weighted child entropy {entropy(y)-gain:.3f}")

best threshold: 0.026   information gain: 0.635 bits
parent entropy 0.996 -> weighted child entropy 0.361


In [5]:
# sanity check: sklearn's first split on the same single feature should land nearby
from sklearn.tree import DecisionTreeClassifier, export_text
dt = DecisionTreeClassifier(max_depth=1).fit(x.reshape(-1, 1), y)
print(export_text(dt, feature_names=["x"]))
print("sklearn split vs scratch split:", dt.tree_.threshold[0].round(3), "vs", round(thr, 3))

|--- x <= 0.03
|   |--- class: 0
|--- x >  0.03
|   |--- class: 1

sklearn split vs scratch split: 0.027 vs 0.026


**Follow-ups:**
- *Gini vs entropy?* Both measure impurity; Gini = 1 − Σp², entropy = −Σp log₂ p; they pick nearly the same splits, Gini is cheaper (no log). Say that, then "Gini is the sklearn default."
- *Why information gain biases toward many-valued features?* A feature with unique values per row can split perfectly (e.g., an ID column) — gain ratio corrects for this by penalizing split multiplicity.
- *Stopping criteria?* Max depth, min samples per split, min gain, pure node — the overfitting guards.

## 4. Logistic Regression From Scratch (with Gradient Descent)

**Plan:** sigmoid(w·x + b); cross-entropy loss; gradient = Xᵀ(ŷ − y)/n; update with learning rate; iterate. This one is *the* most asked because it quietly tests matrix gradients, a loss function, and a training loop at once.

In [6]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def logistic_regression(X, y, lr=0.1, epochs=2_000):
    n, d = X.shape
    w, b = np.zeros(d), 0.0
    for _ in range(epochs):
        p = sigmoid(X @ w + b)
        grad_w = X.T @ (p - y) / n          # cross-entropy gradient
        grad_b = (p - y).mean()
        w, b = w - lr * grad_w, b - lr * grad_b
    return w, b

# standardized features make plain GD converge cleanly
Xc = (iris.data - iris.data.mean(0)) / iris.data.std(0)
mask = iris.target < 2                       # binary problem
Xb, yb = Xc[mask], iris.target[mask]

w, b = logistic_regression(Xb, yb)
pred = (sigmoid(Xb @ w + b) >= 0.5).astype(int)
print(f"scratch accuracy: {(pred == yb).mean():.1%}  weights: {w.round(2)} bias: {b:.2f}")

from sklearn.linear_model import LogisticRegression
skl = LogisticRegression(max_iter=2000).fit(Xb, yb)
print(f"sklearn accuracy : {(skl.predict(Xb) == yb).mean():.1%}  weights: {skl.coef_[0].round(2)} bias: {skl.intercept_[0]:.2f}")
print("coefficients point the same way:", bool((np.sign(w) == np.sign(skl.coef_[0])).all()))

scratch accuracy: 100.0%  weights: [ 1.33 -2.24  2.64  2.27] bias: 2.55
sklearn accuracy : 100.0%  weights: [ 1.04 -1.23  1.74  1.59] bias: 2.40
coefficients point the same way: True


**Follow-ups:**
- *Why cross-entropy not MSE?* MSE + sigmoid gives a non-convex loss with vanishing gradients; cross-entropy keeps the gradient = ŷ − y — convex in w for linear models.
- *Why standardize?* GD moves one step size for all coordinates; unequal scales make some directions crawl (condition number of X, again).
- *Class imbalance?* `class_weight='balanced'` / reweight the loss — and switch metrics to PR-AUC (covered in the metrics notebook).

## 5. The Metrics They Make You Code: ROC-AUC From Scratch

**Q: Compute AUC without sklearn.** The trick: AUC = **probability a random positive scores above a random negative**. Compute all pairwise comparisons (O(n₊·n₋)) or, equivalently and faster, the rank formula: AUC = (Σ ranks of positives − n₊(n₊+1)/2) / (n₊·n₋).

In [7]:
from scipy.stats import rankdata

def auc_score(y_true, scores):
    # rankdata with method="average" gives MID-ranks to tied scores —
    # ties count as half-wins, which is exactly how sklearn treats them
    ranks = rankdata(scores, method="average")
    n_pos = y_true.sum(); n_neg = len(y_true) - n_pos
    return (ranks[y_true == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)

rng = np.random.default_rng(5)
y = rng.integers(0, 2, 400)
scores = np.clip(y * 0.6 + rng.normal(0.5, 0.35, 400), 0, 1)   # positives score higher; clip creates ties

from sklearn.metrics import roc_auc_score
print(f"scratch AUC: {auc_score(y, scores):.4f}")
print(f"sklearn AUC: {roc_auc_score(y, scores):.4f}")
assert abs(auc_score(y, scores) - roc_auc_score(y, scores)) < 1e-12
print("exact match — and note: WITHOUT average ranks for ties the formula would be off by a hair (ties = half-wins)")

scratch AUC: 0.8584
sklearn AUC: 0.8584
exact match — and note: WITHOUT average ranks for ties the formula would be off by a hair (ties = half-wins)


## 6. The Interview Loop for Implementing Anything From Scratch

The same six beats every time — rehearse them as a ritual:

1. **Restate and scope:** "k-means, k fixed, Euclidean, converge on no reassignment — correct?"
2. **Name inputs/outputs and the data structure** before writing: shapes, dtypes.
3. **Write the core loop first**, stub the helpers (`distances`, `assign`), fill them in after.
4. **Test with a tiny hand-checkable case** — 6 points, 2 clusters, run one iteration mentally.
5. **State complexity and failure modes** (local optima, empty clusters, feature scaling).
6. **Offer the validation** — "I'd check my implementation against sklearn on the same data" — which is exactly what this notebook does.

## Pattern Summary Table

| Algorithm | Core equation | Complexity | Classic follow-up |
| --- | --- | --- | --- |
| K-means | argmin ‖x−c‖², mean per cluster | O(n·k·d·iters) | choosing k, k-means++, local optima |
| KNN | majority of k nearest | O(n·d) per query | scaling, distance metric, KD-trees |
| Tree split | max information gain / min Gini | O(n·d·log n) per feature | Gini vs entropy, gain ratio bias |
| Logistic regression | σ(w·x+b), grad = Xᵀ(ŷ−y)/n | O(n·d·epochs) | cross-entropy vs MSE, scaling |
| AUC | P(score₊ > score₋) via ranks | O(n log n) | ROC vs PR under imbalance |